In [ ]:
!pip install --upgrade unsloth unsloth_zoo
!pip install -U torchvision

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive', force_remount=True)
FOLDERNAME = 'Colab Notebooks'
%cd drive/MyDrive/$FOLDERNAME/
train_df = pd.read_csv(f"Peter/data/train_data.csv")
test_df = pd.read_csv(f"Peter/data/test_data.csv")

In [ ]:
from unsloth import FastLanguageModel
import torch
import os
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LlamaForSequenceClassification
from trl import SFTConfig, SFTTrainer
import numpy as np

# Model Name
model_path = "unsloth/Llama-3.2-3B-Instruct"

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
seed = 52
device = "cuda" if torch.cuda.is_available() else "cpu"

prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [ ]:
from datasets import load_dataset, Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import standardize_sharegpt

# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "qwen-2.5",
# )
user_prompt = """
Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes

2) {negative_example_1}
Violation: No

3) {negative_example_2}
Violation: No

4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation: """

classes = ['No', 'Yes']

columns = ['subreddit', 'rule', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2', 'body', 'rule_violation']
dataset = [
    [{"role": "system", "content": prompt},
    {"role": "user", "content": user_prompt.format(subreddit=subreddit,
                                                   rule=rule,
                                                   positive_example_1=positive_example_1,
                                                   positive_example_2=positive_example_2,
                                                   negative_example_1=negative_example_1,
                                                   negative_example_2=negative_example_2,
                                                   body=body)},
    {"role": "assistant", "content": classes[target]}]
    for subreddit, rule, positive_example_1, positive_example_2, negative_example_1, negative_example_2, body, target  in train_df[columns].values]

def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return Dataset.from_dict({'text': texts})

dataset = formatting(dataset)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 4,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>",
    response_part = "<|start_header_id|>assistant<|end_header_id|>",
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from google.colab import userdata
from huggingface_hub import login
HF_key = userdata.get('PLo_HF')
login(token = HF_key)

model_save = '0824-Llama-3-2-3B-Instruct-3E'

model.save_pretrained_merged("0824-Llama-3-2-3B-Instruct-16bit-3E", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged("awilliam60412/0824-Llama-3-2-3B-Instruct-16bit-3E", tokenizer, save_method = "merged_16bit", token = HF_key)